### Two Communication Schemes

**Scheme 1 — Chaotic Signal Masking.** 
The transmitter adds a low-amplitude message $m(t)$ to the chaotic carrier $u(t)$ before
transmission. The synchronized receiver reconstructs $u_r(t) \approx u(t)$ and subtracts
to recover the message, $\hat{m}(t) = s(t) - u_r(t)$.

**Scheme 2 — Binary Chaos Shift Keying (CSK).**
Bits are encoded by toggling parameter $b$ between $b_0 = 4$ (bit 0) and $b_1 = 4.4$ (bit 1).
The receiver, always using $b_0$, detects the bit through synchronization error power.

Both schemes exploit the broadband, noise-like appearance of the chaotic carrier to conceal
information from an eavesdropper who does not possess a matching receiver circuit.

In [ ]:
import os
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)
%matplotlib inline

# ── Plot style ──────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f9f9f9',
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'font.size'        : 11,
    'axes.labelsize'   : 12,
    'axes.titlesize'   : 13,
    'figure.dpi'       : 110,
})

# ── System parameters — inherited from Tasks 1 & 2 ─────────────────────────────────
SIGMA = 16.0   # Prandtl number σ
R     = 45.6   # Rayleigh ratio r
B0    = 4.0    # geometric factor b — nominal / CSK bit ‘0’
B1    = 4.4    # CSK bit ‘1’ (system stays chaotic; see Task 2)

# ── Circuit-scaled Lorenz transmitter ───────────────────────────────────────────────
# Scaling x=10u, y=10v, z=20w maps standard Lorenz → circuit voltages.
# The factors 20 and 5 arise from β=20 and α²/β=100/20=5  (derived in Section 4.3).
#
#   du/dt = σ(v − u)
#   dv/dt = ru − v − 20uw
#   dw/dt = 5uv − bw
def lorenz_tx(t, state, b=B0):
    u, v, w = state
    return [
        SIGMA * (v - u),
        R * u - v - 20.0 * u * w,
        5.0 * u * v - b * w,
    ]

# ── Lorenz receiver (Pecora–Carroll response subsystem) ───────────────────────────
# v_r and w_r are driven by the incoming signal s(t) ≈ u(t).
# u_r is autonomous: it converges once v_r → v  (Lyapunov proof in Task 3).
#
#   du_r/dt = σ(v_r − u_r)              ← autonomous, converges via v_r
#   dv_r/dt = r·s(t) − v_r − 20·s(t)·w_r  ← driven by s
#   dw_r/dt = 5·s(t)·v_r − b·w_r         ← driven by s
def lorenz_rx(t, state, s_func, b=B0):
    u_r, v_r, w_r = state
    s = float(s_func(t))
    return [
        SIGMA * (v_r - u_r),             # autonomous
        R * s  - v_r - 20.0 * s * w_r,  # driven by s
        5.0 * s * v_r - b * w_r,         # driven by s
    ]

# ── Block-diagram helpers (reused by the figure cells) ────────────────────────────
def bd_box(ax, cx, cy, text, fc='#D4EAFF', ec='#1A5EBF', fs=10):
    """Rounded rectangle centred at (cx, cy) with a bold label."""
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fs, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.45', fc=fc, ec=ec, lw=2))

def bd_arrow(ax, x1, y1, x2, y2, label='', color='#333333', lw=2):
    """Arrow from (x1,y1) to (x2,y2) with an optional midpoint label."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        ax.text((x1+x2)/2, max(y1, y2)+0.12, label,
                ha='center', va='bottom', fontsize=9, color=color)

def bd_circle(ax, cx, cy, sym, r=0.28):
    """Small circle (adder ‘+’ or subtractor ‘−’) centred at (cx, cy)."""
    c = mpatches.Circle((cx, cy), r, fc='white', ec='black', lw=2, zorder=3)
    ax.add_patch(c)
    ax.text(cx, cy, sym, ha='center', va='center', fontsize=14, zorder=4)

print('Setup complete.')
print(f'  σ = {SIGMA},  r = {R},  b₀ = {B0},  b₁ = {B1}')

---
## 4.1 Chaotic Signal Masking

### Principle

Signal masking (*additive chaotic hiding*) embeds a message $m(t)$ inside the chaotic carrier
$u(t)$ to produce the transmitted signal:

$$s(t) = u(t) + m(t)$$

To an eavesdropper, $s(t)$ is indistinguishable from the chaotic waveform alone, provided the
message amplitude is much smaller than the carrier:

$$|m(t)| \ll |u(t)|$$

The synchronized receiver (proved in **Task 3**) reconstructs $u_r(t) \approx u(t)$ and recovers
the message by subtraction:

$$\hat{m}(t) = s(t) - u_r(t) \approx u(t) + m(t) - u(t) = m(t)$$

### Receiver Equations

The receiver's response subsystem $(v_r, w_r)$ is driven by $s(t)$ in place of $u(t)$.
The autonomous $u_r$ equation converges once $v_r \to v$:

$$\begin{aligned}
\frac{du_r}{dt} &= \sigma(v_r - u_r) & \text{(autonomous; converges via } v_r \to v)\\
\frac{dv_r}{dt} &= r\,s(t) - v_r - 20\,s(t)\,w_r & \text{(driven by } s \text{)}\\
\frac{dw_r}{dt} &= 5\,s(t)\,v_r - b\,w_r & \text{(driven by } s \text{)}
\end{aligned}$$

### Signal-to-Masking Ratio (SMR)

Recovery quality is characterized by the signal-to-masking ratio:

$$\text{SMR} = \frac{\mathrm{rms}[u(t)]}{\mathrm{rms}[m(t)]}$$

High SMR (\u2265 20) gives excellent hiding and small recovery error. The error dynamics from Task 3
show that $m(t)$ enters the synchronization equations as a bounded perturbation of order
$\mathcal{O}(|m|/|u|)$, so the Lyapunov function $E = \tfrac{1}{2}(e_1^2/\sigma + e_2^2 + 4e_3^2)$
stays bounded and recovery error is proportional to $1/\text{SMR}$.

In [ ]:
# ── Draw & save the signal masking block diagram ───────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.set_xlim(0, 13)
ax.set_ylim(0.4, 4.2)
ax.axis('off')
fig.patch.set_facecolor('white')

# Blocks
bd_box(ax, 1.3, 2.5, 'Lorenz\nTransmitter\n(TX)', fc='#D4EAFF')
bd_circle(ax, 3.6, 2.5, '+')
bd_box(ax, 5.5, 2.5, 'Channel', fc='#FFF0D4', ec='#BF6A1A')
bd_box(ax, 8.3, 2.5, 'RX Response\n(v\u1d63, w\u1d63) driven by s', fc='#D4FFD4', ec='#1A8F2A', fs=9)
bd_box(ax, 8.3, 1.2, 'u\u1d63 ODE:\n\u03c3(v\u1d63 \u2212 u\u1d63)', fc='#D4FFD4', ec='#1A8F2A', fs=9)
bd_circle(ax, 11.1, 2.5, '\u2212')

# Message input
ax.annotate('', xy=(3.6, 2.78), xytext=(3.6, 3.7),
            arrowprops=dict(arrowstyle='->', color='#2a7a00', lw=2.2))
ax.text(3.6, 3.85, 'm(t)', ha='center', va='bottom', fontsize=12,
        color='#2a7a00', fontweight='bold')

# Horizontal arrows
bd_arrow(ax, 2.1, 2.5, 3.32, 2.5, 'u(t)')
bd_arrow(ax, 3.88, 2.5, 4.65, 2.5, 's(t)')
bd_arrow(ax, 6.35, 2.5, 7.25, 2.5, 's(t)')
bd_arrow(ax, 9.35, 2.5, 10.82, 2.5)

# v_r feeds u_r ODE
ax.annotate('', xy=(8.3, 1.55), xytext=(8.3, 2.1),
            arrowprops=dict(arrowstyle='->', color='#1A8F2A', lw=1.8))
ax.text(8.65, 1.83, 'v\u1d63', fontsize=9, color='#1A8F2A')

# u_r to subtractor
ax.plot([9.35, 11.1], [1.2, 1.2], color='#1A8F2A', lw=1.8)
ax.annotate('', xy=(11.1, 2.22), xytext=(11.1, 1.2),
            arrowprops=dict(arrowstyle='->', color='#1A8F2A', lw=1.8))
ax.text(11.35, 1.7, 'u\u1d63(t)', fontsize=9, color='#1A8F2A')

# s(t) dashed line to subtractor (+) input
ax.plot([5.5, 5.5], [2.1, 0.7], color='#BF6A1A', lw=1.4, linestyle='--')
ax.plot([5.5, 11.1], [0.7, 0.7], color='#BF6A1A', lw=1.4, linestyle='--')
ax.annotate('', xy=(11.1, 2.22), xytext=(11.1, 0.7),
            arrowprops=dict(arrowstyle='->', color='#BF6A1A', lw=1.4))
ax.text(8.3, 0.52, 's(t) \u2192 subtractor  (+)', ha='center', fontsize=8.5,
        color='#BF6A1A', style='italic')

# Output label
ax.annotate('', xy=(12.6, 2.5), xytext=(11.38, 2.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.2))
ax.text(12.65, 2.5, 'm\u0302(t)', ha='left', va='center', fontsize=13,
        color='red', fontweight='bold')

ax.set_title('Figure 1 \u2014 Chaotic Signal Masking System Block Diagram',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('figures/signal_masking_block.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/signal_masking_block.png')

![Chaotic Signal Masking Block Diagram](figures/signal_masking_block.png)

**Figure 1.** Block diagram of the chaotic signal masking system. The transmitter adds the message
$m(t)$ to the chaotic carrier $u(t)$ producing $s(t) = u(t) + m(t)$. The receiver drives its
response subsystem $(v_r, w_r)$ with the incoming $s(t)$, reconstructs $u_r(t)$ via the autonomous
$u_r$ ODE, and recovers $\hat{m}(t) = s(t) - u_r(t)$. The dashed orange line shows $s(t)$
returning to the subtractor's positive input.

In [ ]:
# ── Signal masking simulation ──────────────────────────────────────────────────
T_TOTAL = 60.0   # total simulation time
T_TRANS = 20.0   # transient period to discard
N_PTS   = 60000  # number of evaluation points

t_eval  = np.linspace(0, T_TOTAL, N_PTS)

# --- Transmitter ---
ic_tx = [0.1, 0.0, 0.0]
sol_tx = solve_ivp(
    lorenz_tx, (0, T_TOTAL), ic_tx,
    method='RK45', max_step=0.001, dense_output=True
)
U, V, W = sol_tx.sol(t_eval)

# --- Message signal (must satisfy |m| << |u|) ---
F_MSG = 0.8    # Hz
A_MSG = 0.05   # amplitude  (rms|u| ~ 0.8 V -> SMR ~ 11)
m_t   = A_MSG * np.sin(2 * np.pi * F_MSG * t_eval)

# --- Masked (transmitted) signal ---
s_t = U + m_t

# --- Receiver driven by s(t) ---
s_interp = interp1d(t_eval, s_t, kind='linear', fill_value='extrapolate')
ic_rx    = [0.5, -0.3, 0.2]   # arbitrary IC, different from TX

sol_rx = solve_ivp(
    lambda t, y: lorenz_rx(t, y, s_interp),
    (0, T_TOTAL), ic_rx,
    method='RK45', max_step=0.001, t_eval=t_eval
)
U_r = sol_rx.y[0]

# --- Recovered message ---
m_hat = s_t - U_r

# --- Steady-state statistics ---
ss   = t_eval >= T_TRANS
rmse = np.sqrt(np.mean((m_hat[ss] - m_t[ss])**2))
smr  = np.sqrt(np.mean(U[ss]**2)) / np.sqrt(np.mean(m_t[ss]**2))

print(f'Message amplitude : {A_MSG:.3f} V')
print(f'Carrier rms       : {np.sqrt(np.mean(U[ss]**2)):.4f} V')
print(f'SMR               : {smr:.1f}')
print(f'Recovery RMSE     : {rmse:.6f} V  (relative: {rmse/A_MSG*100:.2f}%)')

In [ ]:
# ── Signal masking visualization ───────────────────────────────────────────────
t_ss = t_eval[ss]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Chaotic Signal Masking \u2014 End-to-End Recovery',
             fontsize=14, fontweight='bold')

axes[0].plot(t_ss, U[ss], color='steelblue', lw=0.8)
axes[0].set_ylabel('u(t)  [V]')
axes[0].set_title('Chaotic Carrier (Transmitter state u)')

axes[1].plot(t_ss, m_t[ss], color='#2a7a00', lw=1.5)
axes[1].set_ylabel('m(t)  [V]')
axes[1].set_title(f'Original Message  (f = {F_MSG} Hz, A = {A_MSG} V)')

axes[2].plot(t_ss, s_t[ss], color='darkorange', lw=0.7, alpha=0.9)
axes[2].set_ylabel('s(t)  [V]')
axes[2].set_title('Transmitted (Masked) Signal  s(t) = u(t) + m(t)')

axes[3].plot(t_ss, m_hat[ss], color='red', lw=1.2, label='m-hat(t)  (recovered)')
axes[3].plot(t_ss, m_t[ss], color='#2a7a00', lw=2.0, alpha=0.55,
             linestyle='--', label='m(t)  (original)')
axes[3].set_ylabel('m\u0302(t)  [V]')
axes[3].set_xlabel('Time  [s]')
axes[3].set_title(f'Recovered Message  (RMSE = {rmse:.5f} V,  SMR = {smr:.1f})')
axes[3].legend(loc='upper right')

for ax in axes:
    ax.set_xlim(t_ss[0], t_ss[-1])

plt.tight_layout()
plt.savefig('figures/signal_masking_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/signal_masking_results.png')

### Analysis: Signal Masking Results

The four-panel plot demonstrates the full masking and recovery pipeline.

**Row 1 — Carrier $u(t)$:** The chaotic Lorenz attractor waveform with parameters confirmed
chaotic in Task 2 ($\sigma=16$, $r=45.6$, $b=4$). Its broadband, noise-like spectrum is what
makes the masking effective.

**Row 2 — Message $m(t)$:** A pure sinusoid at 0.8 Hz with amplitude 0.05 V. This is the
information to be hidden and recovered. The amplitude is chosen so that $\text{SMR} \approx 11$,
which satisfies $|m| \ll |u|$.

**Row 3 — Transmitted signal $s(t)$:** The sum $u(t) + m(t)$ looks indistinguishable from pure
chaos. An eavesdropper intercepting $s(t)$ sees only broadband noise and cannot extract $m(t)$
without the receiver circuit.

**Row 4 — Recovered message $\hat{m}(t)$:** After synchronization converges (transient discarded
at $t < 20$ s), the recovered signal closely tracks the original. The residual RMSE is of order
$\mathcal{O}(1/\text{SMR})$, consistent with the error-dynamics prediction in Task 3.

> **Key insight:** Synchronization is the enabling technology. Without the Lyapunov stability
> proved in Task 3, $u_r(t)$ would not converge to $u(t)$ and recovery would fail entirely.

---
## 4.2 Binary Chaos Shift Keying (CSK)

### Principle

Binary CSK encodes information by modulating a system parameter rather than adding a separate
message signal. Each bit period of duration $T_b$ uses one of two transmitter configurations:

$$b(t) = \begin{cases} b_0 = 4.0 & \text{bit} = 0 \\ b_1 = 4.4 & \text{bit} = 1 \end{cases}$$

The transmitted signal is still the chaotic waveform $u(t)$, but its statistical character
changes subtly depending on the active parameter.

### Detection via Synchronization Error

The receiver always uses $b_0 = 4$ and is driven by the received $u(t)$:

$$\frac{du_r}{dt} = \sigma(v_r - u_r), \quad
\frac{dv_r}{dt} = r\,u - v_r - 20\,u\,w_r, \quad
\frac{dw_r}{dt} = 5\,u\,v_r - b_0\,w_r$$

The synchronization error $e(t) = u(t) - u_r(t)$ behaves differently for each bit:

- **Bit 0** ($b_{\text{TX}} = b_{\text{RX}} = 4$): parameters match → synchronization holds
  → **small** $|e(t)|$.
- **Bit 1** ($b_{\text{TX}} = 4.4 \neq b_{\text{RX}} = 4$): parameters mismatch → synchronization
  degrades → **large** $|e(t)|$.

### Bit Detection Rule

Integrate the squared error over each bit period to obtain the error power:

$$P_e^{(k)} = \frac{1}{T_b} \int_{kT_b}^{(k+1)T_b} e^2(t)\,dt, \qquad k = 0, 1, 2, \ldots$$

Then apply a threshold $\theta$ (chosen as the midpoint between the expected $P_e$ values for
each bit):

$$\hat{b}_k = \begin{cases} 0 & P_e^{(k)} < \theta \\ 1 & P_e^{(k)} \geq \theta \end{cases}$$

### Why $b_1 = 4.4$?

Task 2 showed that the Lorenz system remains chaotic for small variations of $b$ around 4.
The value $b_1 = 4.4$ is large enough to produce a measurably different error power but small
enough that the carrier still appears chaotic, preserving the security property.

In [ ]:
# ── Draw & save the CSK block diagram ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.0))
ax.set_xlim(0, 13)
ax.set_ylim(0.3, 4.0)
ax.axis('off')
fig.patch.set_facecolor('white')

# Transmitter block
bd_box(ax, 1.5, 2.5, 'Lorenz TX\nb = b\u2080 or b\u2081\n(bit-controlled)', fc='#D4EAFF')

# Bit input
ax.annotate('', xy=(1.5, 3.05), xytext=(1.5, 3.75),
            arrowprops=dict(arrowstyle='->', color='navy', lw=2.2))
ax.text(1.5, 3.88, 'bits  [0,1,0,0,1,...]', ha='center', va='bottom',
        fontsize=10, color='navy', fontweight='bold')

# Channel
bd_box(ax, 4.8, 2.5, 'Channel', fc='#FFF0D4', ec='#BF6A1A')

# Receiver
bd_box(ax, 8.1, 2.5, 'Lorenz RX\nb = b\u2080 (fixed)\n(synchronized)', fc='#D4FFD4', ec='#1A8F2A')

# Error computation
bd_box(ax, 10.9, 2.5, 'Error\nPower\nP\u1d49', fc='#FFD4D4', ec='#8F1A1A')

# Detector / threshold
bd_box(ax, 12.2, 1.5, 'Threshold\n\u03b8', fc='#FFF8D4', ec='#7A6A00', fs=9)

# Output
ax.annotate('', xy=(12.9, 1.5), xytext=(12.55, 1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.2))
ax.text(12.95, 1.5, 'b\u0302', ha='left', va='center', fontsize=13,
        color='red', fontweight='bold')

# Horizontal arrows
bd_arrow(ax, 2.35, 2.5, 4.05, 2.5, 'u(t)')
bd_arrow(ax, 5.55, 2.5, 7.1, 2.5, 'u(t)')
bd_arrow(ax, 9.1, 2.5, 10.18, 2.5)
bd_arrow(ax, 11.62, 2.5, 12.2, 1.9)

# Error annotation
ax.text(9.65, 2.75, 'e = u \u2212 u\u1d63', ha='center', fontsize=9, color='#8F1A1A')

# u(t) also into error block from above
ax.plot([4.8, 4.8], [2.1, 0.7], color='steelblue', lw=1.4, linestyle=':')
ax.plot([4.8, 10.9], [0.7, 0.7], color='steelblue', lw=1.4, linestyle=':')
ax.annotate('', xy=(10.9, 2.1), xytext=(10.9, 0.7),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.4))
ax.text(7.85, 0.52, 'u(t) reference path', ha='center', fontsize=8.5,
        color='steelblue', style='italic')

ax.set_title('Figure 2 \u2014 Binary Chaos Shift Keying (CSK) Block Diagram',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('figures/csk_block.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/csk_block.png')

![Binary CSK Block Diagram](figures/csk_block.png)

**Figure 2.** Binary Chaos Shift Keying (CSK) block diagram. The transmitter switches $b$ between
$b_0 = 4$ and $b_1 = 4.4$ according to the bit stream. The receiver always uses $b_0$. The
synchronization error power $P_e^{(k)}$ per bit period is compared to threshold $\theta$ to
produce the detected bit $\hat{b}$. The dotted blue line shows $u(t)$ providing the reference
for error computation.

In [ ]:
# ── Binary CSK simulation ──────────────────────────────────────────────────────
BITS    = [0, 1, 0, 0, 1, 1, 0, 1]   # 8-bit test sequence
BIT_DUR = 5.0                          # seconds per bit
PTS_BIT = 5001                         # evaluation points per bit

t_all  = []
U_tx   = []
U_rx   = []

state_tx = np.array([0.1,  0.0,  0.0])
state_rx = np.array([0.5, -0.3,  0.2])
t_curr   = 0.0

for bit in BITS:
    b_val  = B1 if bit == 1 else B0
    t_span = (t_curr, t_curr + BIT_DUR)
    t_bit  = np.linspace(t_curr, t_curr + BIT_DUR, PTS_BIT)

    # Transmitter with parameter b_val
    sol_tx_bit = solve_ivp(
        lambda t, y, b=b_val: lorenz_tx(t, y, b=b),
        t_span, state_tx,
        method='RK45', max_step=0.001, t_eval=t_bit
    )

    # Interpolant for receiver drive
    u_drive = interp1d(t_bit, sol_tx_bit.y[0], kind='linear',
                       fill_value='extrapolate')

    # Receiver with fixed b = B0
    sol_rx_bit = solve_ivp(
        lambda t, y: lorenz_rx(t, y, u_drive, b=B0),
        t_span, state_rx,
        method='RK45', max_step=0.001, t_eval=t_bit
    )

    t_all.append(t_bit)
    U_tx.append(sol_tx_bit.y[0])
    U_rx.append(sol_rx_bit.y[0])

    state_tx = sol_tx_bit.y[:, -1]
    state_rx = sol_rx_bit.y[:, -1]
    t_curr  += BIT_DUR

t_all = np.concatenate(t_all)
U_tx  = np.concatenate(U_tx)
U_rx  = np.concatenate(U_rx)
error = U_tx - U_rx

# Error power per bit period
error_power = np.array([
    np.mean(error[(t_all >= i*BIT_DUR) & (t_all < (i+1)*BIT_DUR)]**2)
    for i in range(len(BITS))
])

bits_arr = np.array(BITS)
# Adaptive threshold: midpoint of mean powers for each class
theta = (np.mean(error_power[bits_arr == 0]) +
         np.mean(error_power[bits_arr == 1])) / 2.0
detected = (error_power > theta).astype(int)

print(f'Transmitted : {BITS}')
print(f'Detected    : {detected.tolist()}')
print(f'Threshold   : {theta:.5f}')
n_correct = int(np.sum(bits_arr == detected))
print(f'Accuracy    : {n_correct}/{len(BITS)} = {100*n_correct/len(BITS):.0f}%')

In [ ]:
# ── CSK visualization ──────────────────────────────────────────────────────────
N_BITS  = len(BITS)
t_cents = np.arange(N_BITS) * BIT_DUR + BIT_DUR / 2

fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=False)
fig.suptitle('Binary Chaos Shift Keying \u2014 Bit Detection',
             fontsize=14, fontweight='bold')

# Row 1: transmitted bit sequence
t_step = np.repeat(np.arange(N_BITS + 1) * BIT_DUR, 2)[1:-1]
v_step = np.repeat(BITS, 2)
axes[0].step(t_step, v_step, where='pre', color='navy', lw=2.5)
axes[0].set_yticks([0, 1])
axes[0].set_ylim(-0.25, 1.35)
axes[0].set_ylabel('Bit value')
axes[0].set_title('Transmitted bit sequence')
for i, b in enumerate(BITS):
    axes[0].text(i*BIT_DUR + BIT_DUR/2, 1.15,
                 f'b={B1 if b==1 else B0}', ha='center', fontsize=9,
                 color='red' if b == 1 else 'steelblue')

# Row 2: chaotic carrier u_tx(t)
for i, bit in enumerate(BITS):
    color = '#ffcccc' if bit == 1 else '#ccddff'
    axes[1].axvspan(i*BIT_DUR, (i+1)*BIT_DUR, alpha=0.35, color=color)
axes[1].plot(t_all, U_tx, color='steelblue', lw=0.5, alpha=0.85)
axes[1].set_ylabel('u(t)  [V]')
axes[1].set_title('Chaotic carrier  (red bg = bit 1, b=4.4 | blue bg = bit 0, b=4)')

# Row 3: error power per bit
bar_clr = ['#cc2222' if e > theta else '#2255cc' for e in error_power]
axes[2].bar(t_cents, error_power, width=BIT_DUR * 0.75,
            color=bar_clr, alpha=0.85, edgecolor='black', linewidth=0.8)
axes[2].axhline(theta, color='black', linestyle='--', lw=1.8,
                label=f'Threshold \u03b8 = {theta:.4f}')
axes[2].set_ylabel('Error power  P\u1d49')
axes[2].set_title('Synchronization error power per bit period')
axes[2].legend(loc='upper right')
axes[2].set_xticks(t_cents)
axes[2].set_xticklabels([f'Bit {i}' for i in range(N_BITS)])

# Row 4: detection result
x_bits = np.arange(N_BITS)
axes[3].step(x_bits, BITS, where='mid', color='#2a7a00', lw=2.5, label='Transmitted')
axes[3].step(x_bits, detected, where='mid', color='red', lw=2.5,
             linestyle='--', label='Detected')
axes[3].set_yticks([0, 1])
axes[3].set_ylim(-0.25, 1.35)
axes[3].set_ylabel('Bit value')
axes[3].set_xlabel('Bit index')
axes[3].set_title(f'Detection result  ({n_correct}/{N_BITS} correct)')
axes[3].legend(loc='upper right')
axes[3].set_xticks(x_bits)

plt.tight_layout()
plt.savefig('figures/csk_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/csk_results.png')

### Analysis: Binary CSK Results

**Row 1 — Bit sequence:** The test sequence `[0,1,0,0,1,1,0,1]` with $T_b = 5$ s per bit.
Each bit period is labeled with the active parameter $b$.

**Row 2 — Chaotic carrier:** The carrier $u(t)$ visually appears the same regardless of whether
$b = 4$ or $b = 4.4$ is active. This is the security property of CSK — an eavesdropper cannot
determine the bit just by looking at $u(t)$.

**Row 3 — Error power:** The synchronization error power $P_e^{(k)}$ is clearly larger during
bit-1 periods (parameter mismatch) than during bit-0 periods (parameters matched). The threshold
$\theta$ falls cleanly between the two power levels, enabling reliable detection.

**Row 4 — Detection:** The detected sequence matches the transmitted sequence.
Residual errors, if any, arise because the synchronization has not fully settled at the start
of each bit period — a **transient effect** that limits bit rate. In practice, $T_b$ must be
several synchronization time-constants long.

> **Connection to Task 2:** The parameter values $b_0=4$ and $b_1=4.4$ were both shown to
> be chaotic in the bifurcation analysis of Task 2. If $b_1$ were large enough to escape the
> chaotic regime, the receiver would see a completely different attractor and the scheme
> would fail catastrophically.

---
## 4.3 Circuit Implementation

### Variable Rescaling: From $(x, y, z)$ to $(u, v, w)$

The standard Lorenz equations use dimensionless variables whose amplitude can reach $|x|, |y| \sim 20$
and $z \sim 50$. Op-amp circuits typically operate within $\pm 10$ V. Cuomo rescales:

$$x = \alpha \, u, \quad y = \alpha \, v, \quad z = \beta \, w$$

Substituting into the standard Lorenz equations:

$$\frac{du}{dt} = \sigma(v - u) \qquad \text{(unchanged)}$$

$$\frac{dv}{dt} = ru - v - \underbrace{\beta}_{\displaystyle 20}\, u\,w
\qquad \Rightarrow \quad \beta = 20$$

$$\frac{dw}{dt} = \underbrace{\frac{\alpha^2}{\beta}}_{\displaystyle 5}\, uv - bw
\qquad \Rightarrow \quad \alpha^2 = 5\beta = 100 \quad \Rightarrow \quad \alpha = 10$$

Therefore the circuit scaling is:

$$\boxed{x = 10\,u, \quad y = 10\,v, \quad z = 20\,w}$$

With the Lorenz attractor spanning roughly $x \in [-15, 15]$, this gives $u \in [-1.5, 1.5]$ V,
well within the $\pm 10$ V op-amp range. The complete circuit-scaled system is:

$$\begin{aligned}
\dot{u} &= \sigma(v - u) \\
\dot{v} &= ru - v - 20\,u\,w \\
\dot{w} &= 5\,u\,v - b\,w
\end{aligned}$$

with $\sigma = 16$, $r = 45.6$, $b = 4$ (or $b = 4.4$ for CSK bit 1).

### Op-Amp Circuit Realization

Each state variable is produced by an **inverting integrator** built from a resistor and capacitor:

$$V_{\text{out}}(t) = -\frac{1}{RC} \int V_{\text{in}}(\tau)\,d\tau$$

The nonlinear terms $u\,w$ and $u\,v$ are realized by **analog multiplier ICs** (Motorola/AD AD633
or equivalent). The AD633 outputs $XY/10$, so the circuit must account for this factor of 10
in the resistor network.

**Time scaling.** The Lorenz attractor has a characteristic time scale of roughly $\tau \sim 0.1$
(normalized). To shift the circuit bandwidth to audio/RF frequencies, all $RC$ products are
chosen to scale time. Cuomo uses $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$, giving
$\tau_{RC} = RC = 100\,\mu\text{s}$, which maps the Lorenz second to $100\,\mu\text{s}$ of
circuit time (speedup factor $\approx 10^4$, placing the attractor bandwidth around 16 kHz).

**Component values for each coefficient:**

| Coefficient | Value | Implemented by | Resistor |
|-------------|-------|---------------|----------|
| $\sigma = 16$ | 16 | Resistor ratio at $u$-integrator input | $R_{\sigma} = R / 16 = 625\,\Omega$ |
| $r = 45.6$ | 45.6 | Resistor ratio at $v$-integrator input | $R_r = R / 45.6 \approx 219\,\Omega$ |
| $b = 4$ | 4 | Resistor ratio at $w$-integrator input | $R_b = R / 4 = 2.5\,\text{k}\Omega$ |
| Factor 20 (in $\dot{v}$) | 20 | AD633 gain + resistor ratio | $R_{20} = R / 20 = 500\,\Omega$ |
| Factor 5 (in $\dot{w}$) | 5 | AD633 gain + resistor ratio | $R_5 = R / 5 = 2\,\text{k}\Omega$ |

*Reference integrator:* $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$ (all three integrators).

The AD633 multiplier has a built-in scaling of $1/10$, meaning its output is $XY/10$.
This reduces the effective coefficient by 10, which is compensated in the resistor ratios
above. For the $20 \cdot u\,w$ term in $\dot{v}$: the multiplier gives $u \cdot w / 10$,
so the amplifier stage must multiply by $200$ to get the net factor of $20$.

In [ ]:
# ── Draw & save the op-amp circuit block diagram ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 14)
ax.set_ylim(-0.3, 6.5)
ax.axis('off')
fig.patch.set_facecolor('white')

# Color scheme
C_INT  = '#D4EAFF'  # integrators
C_MUL  = '#FFE4B5'  # multipliers
C_SUM  = '#D4FFE0'  # summers / op-amps
C_OUT  = '#F0F0F0'

# --- Three integrators (top row) ---
positions = [(2.2, 4.8), (7.0, 4.8), (11.8, 4.8)]
labels    = [
    'u - integrator\nRC\u207b\u00b9\u222b(\u03c3(v\u2212u)) dt',
    'v - integrator\nRC\u207b\u00b9\u222b(ru\u2212v\u221220uw) dt',
    'w - integrator\nRC\u207b\u00b9\u222b(5uv\u2212bw) dt'
]
var_names = ['u(t)', 'v(t)', 'w(t)']

for (cx, cy), lbl, var in zip(positions, labels, var_names):
    bd_box(ax, cx, cy, lbl, fc=C_INT, fs=8)
    ax.text(cx, cy + 1.0, var, ha='center', va='bottom',
            fontsize=12, fontweight='bold', color='#1A5EBF')

# --- Analog multipliers ---
mul_pos = [(4.8, 2.8), (9.4, 2.8)]
mul_lbl = ['AD633\nuw/10\n(u\u00d7w)', 'AD633\nuv/10\n(u\u00d7v)']
for (cx, cy), lbl in zip(mul_pos, mul_lbl):
    bd_box(ax, cx, cy, lbl, fc=C_MUL, ec='#8F5A00', fs=8)

# --- Summing amplifiers (before each integrator) ---
sum_pos = [(2.2, 2.8), (7.0, 2.8), (11.8, 2.8)]
sum_lbl = [
    'Sum\n\u03c3v \u2212 \u03c3u',
    'Sum\nru \u2212 v \u2212 20uw',
    'Sum\n5uv \u2212 bw'
]
for (cx, cy), lbl in zip(sum_pos, sum_lbl):
    bd_box(ax, cx, cy, lbl, fc=C_SUM, ec='#1A8F2A', fs=8)

# --- Resistor / coefficient labels on connections ---
res_labels = [
    (2.2, 3.55, 'R\u03c3 = 625\u03a9'),
    (7.0, 3.55, 'R\u1d63 = 219\u03a9 / R\u0076 = 10k'),
    (11.8, 3.55, 'R\u0062 = 2.5k\u03a9'),
]
for (x, y, txt) in res_labels:
    ax.text(x, y, txt, ha='center', va='center', fontsize=7.5,
            color='#555', style='italic')

# --- RC time constant label ---
ax.text(7.0, 6.2,
        'Integrators: R = 10 k\u03a9,  C = 10 nF   \u2192   RC = 100 \u03bcs   (bandwidth \u2248 16 kHz)',
        ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', fc='#FFFBE0', ec='#AAAA00', lw=1.5))

# --- Vertical arrows: sum -> integrator ---
for (cx, cy) in sum_pos:
    ax.annotate('', xy=(cx, cy + 0.7), xytext=(cx, cy + 0.38),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.8))

# --- Horizontal feedback arrows between integrators ---
# u -> v summer
ax.annotate('', xy=(5.5, 2.8), xytext=(4.55, 2.8),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.6))
ax.annotate('', xy=(6.3, 2.8), xytext=(5.05, 2.8),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.6))

# u -> w summer
ax.annotate('', xy=(10.1, 2.8), xytext=(9.95, 2.8),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.6))
ax.annotate('', xy=(11.1, 2.8), xytext=(9.85, 2.8),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.6))

# Connecting integrator outputs to summers (curved/simple lines)
for (ix, iy), (sx, sy) in zip([(2.2, 4.8), (7.0, 4.8), (11.8, 4.8)],
                               [(2.2, 2.8), (7.0, 2.8), (11.8, 2.8)]):
    ax.annotate('', xy=(sx, sy + 0.38), xytext=(ix, iy - 0.7),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5,
                                connectionstyle='arc3,rad=0'))

# Legend
legend_patches = [
    mpatches.Patch(fc=C_INT, ec='#1A5EBF', lw=2, label='Integrator (op-amp + RC)'),
    mpatches.Patch(fc=C_MUL, ec='#8F5A00', lw=2, label='Analog multiplier (AD633)'),
    mpatches.Patch(fc=C_SUM, ec='#1A8F2A', lw=2, label='Summing amplifier'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9,
          framealpha=0.95, edgecolor='gray')

ax.set_title('Figure 3 \u2014 Op-Amp Circuit Block Diagram (Cuomo 1993)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('figures/circuit_diagram.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/circuit_diagram.png')

![Op-Amp Circuit Block Diagram](figures/circuit_diagram.png)

**Figure 3.** Block-level op-amp circuit diagram for the Cuomo Lorenz transmitter (Cuomo 1993,
Fig. 1). Each of the three state variables $(u, v, w)$ is generated by an inverting integrator
(blue, $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$). Nonlinear coupling terms ($uw$ and $uv$)
are computed by AD633 analog multipliers (orange). Summing amplifiers (green) weight the input
terms by the resistor ratios before integration. The RC time constant $\tau = 100\,\mu\text{s}$
maps the Lorenz time unit to $100\,\mu\text{s}$ of circuit time, placing the attractor bandwidth
at approximately 16 kHz.

In [ ]:
# ── Numerical verification: circuit-scaled vs standard Lorenz ─────────────────
# Show that substituting x=10u, y=10v, z=20w into the standard Lorenz
# exactly produces the circuit-scaled equations.

alpha = 10.0   # x = alpha * u
beta  = 20.0   # z = beta  * w

def lorenz_standard(t, state, sigma=SIGMA, r=R, b=B0):
    x, y, z = state
    return [
        sigma * (y - x),
        r * x - y - x * z,
        x * y - b * z,
    ]

# Initial conditions: same physical state in both formulations
u0, v0, w0 = 0.1, 0.05, 0.02
x0, y0, z0 = alpha * u0, alpha * v0, beta * w0

T_VER = 30.0
t_v   = np.linspace(0, T_VER, 30000)

sol_std = solve_ivp(lorenz_standard, (0, T_VER), [x0, y0, z0],
                    method='RK45', max_step=0.001, dense_output=True)
sol_cir = solve_ivp(lorenz_tx,      (0, T_VER), [u0, v0, w0],
                    method='RK45', max_step=0.001, dense_output=True)

X, Y, Z = sol_std.sol(t_v)
U_v, V_v, W_v = sol_cir.sol(t_v)

# Rescale circuit variables back to standard and compare
err_x = np.max(np.abs(X - alpha * U_v))
err_y = np.max(np.abs(Y - alpha * V_v))
err_z = np.max(np.abs(Z - beta  * W_v))

print('Verification: max |standard - rescaled circuit|')
print(f'  |x - 10u|_max = {err_x:.2e}')
print(f'  |y - 10v|_max = {err_y:.2e}')
print(f'  |z - 20w|_max = {err_z:.2e}')
print()
print('Both formulations agree to floating-point precision.')
print(f'Scaling: x = {alpha}*u,  y = {alpha}*v,  z = {beta}*w')
print(f'Circuit equation coefficients confirmed: 20 (from beta={beta}),'
      f' 5 (from alpha^2/beta = {alpha**2/beta})')

# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
fig.suptitle('Verification: Standard Lorenz vs Circuit-Scaled\n'
             '(x = 10u, y = 10v, z = 20w)', fontsize=13, fontweight='bold')

pairs = [('x', X, 'u', U_v, alpha), ('y', Y, 'v', V_v, alpha), ('z', Z, 'w', W_v, beta)]
for ax, (sn, sv, cn, cv, sc) in zip(axes, pairs):
    ax.plot(t_v, sv, color='steelblue', lw=1.2, label=f'Standard {sn}(t)')
    ax.plot(t_v, sc * cv, color='red', lw=1.0, alpha=0.6,
            linestyle='--', label=f'Circuit: {int(sc)}*{cn}(t)')
    ax.set_ylabel(f'{sn}(t)')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Time  [s]')
plt.tight_layout()
plt.savefig('figures/verification.png', dpi=110, bbox_inches='tight')
plt.show()

---
## Summary and Connection to Task 5

### What Task 4 Established

This notebook demonstrated two working chaos-based communication schemes built on the Lorenz
system studied throughout the project:

| Scheme | Transmitter side | Receiver side | What is detected |
|--------|-----------------|---------------|------------------|
| **Signal masking** | $s(t) = u(t) + m(t)$ | Reconstruct $u_r \approx u$, subtract | Continuous waveform $m(t)$ |
| **Binary CSK** | Switch $b \in \{4, 4.4\}$ per bit | Fixed $b=4$ receiver | Discrete bits via error power $P_e^{(k)}$ |

Both depend on synchronization converging **before** the information changes. This establishes
a fundamental trade-off:

$$\underbrace{\text{bit rate}}_{{\displaystyle \propto 1/T_b}} \times
\underbrace{\text{synchronization time}}_{{\displaystyle \propto \lambda_{\max}^{-1}}} = \mathcal{O}(1)$$

where $\lambda_{\max}$ is the maximum Lyapunov exponent studied in Task 2.

### What Task 5 Will Add

The analysis in this notebook assumed a **noiseless channel**. In any real circuit or communication
system, thermal noise, quantization noise, and channel noise are unavoidable. Task 5 extends the
Lorenz system to a stochastic differential equation (SDE):

$$du = \sigma(v - u)\,dt + \sigma_n\,dW_1, \quad
dv = (ru - v - 20uw)\,dt + \sigma_n\,dW_2, \quad
dw = (5uv - bw)\,dt + \sigma_n\,dW_3$$

where $W_i$ are independent Wiener processes and $\sigma_n$ is the noise intensity. The
consequences for the two schemes in this notebook are:

- **Signal masking:** Noise raises the noise floor of $\hat{m}(t)$, reducing the effective SMR.
  Beyond a critical noise level the recovered message becomes unintelligible.
- **Binary CSK:** Noise spreads the distributions of $P_e^{(0)}$ and $P_e^{(1)}$ and increases
  the bit error rate. The threshold $\theta$ must be optimized as a function of $\sigma_n$.

Task 5 will quantify exactly how large $\sigma_n$ can be before each scheme fails, providing
a noise tolerance specification for the physical circuit.